# Import libraries

In [1]:
import pandas as pd
import os
from pathlib import Path
import shutil
from typing import List
import json 
from tqdm import tqdm
import glob
import soundfile as sf

import concurrent.futures
from typing import List, Tuple, Dict
import multiprocessing

import sys
sys.path.append('..')
from utils.audio_util import convert_mp3_to_flac, resample_audios, trim_silence_with_vad, normalize_audio_files
from utils.file_util import recursive_copy
from utils.text_util import clean_text_cv, handle_maiyamok

from transformers import Wav2Vec2FeatureExtractor, WavLMForXVector
import torch
import torchaudio
from collections import defaultdict
import gc
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_distances

/home/ming/Capstone/dubbing-ai/Restructure/converter/../utils/audio_util.py:12: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import SepformerSeparation as separator


# Read the validated tsv

In [2]:
validated_data = pd.read_csv('../data/raw/cv-corpus-20.0-2024-12-06/th/validated.tsv', sep='\t')

/tmp/ipykernel_44860/1241631071.py:1: DtypeWarning: Columns (9,12) have mixed types. Specify dtype option on import or set low_memory=False.
  validated_data = pd.read_csv('../data/raw/cv-corpus-20.0-2024-12-06/th/validated.tsv', sep='\t')


# Define word replacement function

In [3]:
replacing_word = [
    ['เพฃร', 'เพชร'],
]

maiyamok_exceptions = {
    'เธอเป็นคนดีมาก ๆ': 'เธอเป็นคนดีมากมาก',
    'ให้ประสานงานกับสมาชิกคนอื่น ๆ และหารือเกี่ยวกับปัญหานี้ในภายหลัง': 'ให้ประสานงานกับสมาชิกคนอื่นคนอื่น และหารือเกี่ยวกับปัญหานี้ในภายหลัง',
    'ช่างเป็นสวนหลังบ้านที่ดีงามมาก ๆ จริง ๆ คุณนาย !': 'ช่างเป็นสวนหลังบ้านที่ดีงามมากมาก จริงจริง คุณนาย !',
    'ฉันอาจจะผ่านเรื่องเล็ก ๆ น้อย ๆ นั้นไปได้ด้วยดี': 'ฉันอาจจะผ่านเรื่องเล็กเล็ก น้อยน้อย นั้นไปได้ด้วยดี',
    'มันทำให้รู้สึกสดชื่นจริง ๆ ที่ได้นั่งท่ามกลางบรรยากาศที่มีแต่ลมพัดเย็น ๆ ที่แสนสบาย': 'มันทำให้รู้สึกสดชื่นจริงจริง ที่ได้นั่งท่ามกลางบรรยากาศที่มีแต่ลมพัดเย็นเย็น ที่แสนสบาย',
    'ซาลกำลังทำให้คนอื่น ๆ โกรธ': 'ซาลกำลังทำให้คนอื่นอื่น โกรธ',
    'มีทันตแพทย์และแพทย์คนอื่น ๆ ในคลินิกนี้': 'มีทันตแพทย์และแพทย์คนอื่นอื่น ในคลินิกนี้',
    'ใครพูดรัว ๆ ติดกันได้นาน ๆ บอกเราด้วย': 'ใครพูดรัวรัว ติดกันได้นานนาน บอกเราด้วย',
    '“ผมเสียใจจริง ๆ ที่ต้องไปครับท่าน” ฉันตอบกลับ': '“ผมเสียใจจริงจริง ที่ต้องไปครับท่าน” ฉันตอบกลับ',
    'ใจเย็น ๆ ค่ะ ไม่ต้องตื่นเต้น': 'ใจเย็นเย็น ค่ะ ไม่ต้องตื่นเต้น',
    'เรื่องนี้มันดีมาก ๆ': 'เรื่องนี้มันดีมากมาก',
    'คุณผู้หญิงเหมือนเด็กเล็ก ๆ': 'คุณผู้หญิงเหมือนเด็กเล็กเล็ก',
    "'ช่างเป็นดอกไม้ดอกใหญ่ ๆ สมอย่างที่ต้องเป็นจริง ๆ !' เธอแสดงความคิดต่อมาของเธอ": "'ช่างเป็นดอกไม้ดอกใหญ่ใหญ่ สมอย่างที่ต้องเป็นจริงจริง !' เธอแสดงความคิดต่อมาของเธอ",
    'คนอื่น ๆ เดินตามรอยของเขา' :'คนอื่นอื่น เดินตามรอยของเขา'
}

def preprocess_words(dataframe: pd.DataFrame, column_name: str, replacing_pairs: List[List[str]]) -> pd.DataFrame:
    dataframe[column_name] = dataframe[column_name].apply(lambda x: handle_maiyamok(x, maiyamok_exceptions))
    for old_word, new_word in replacing_pairs:
        dataframe[column_name] = dataframe[column_name].apply(lambda x: x.replace(old_word, new_word))
    return dataframe

# Filter and group client_id that have over 100 records

In [4]:
filtered_data = validated_data[
    validated_data['client_id'].map(
        validated_data['client_id'].value_counts() >= 100
    )
]
filtered_data = preprocess_words(filtered_data, 'sentence', replacing_word)
grouped = filtered_data.groupby('client_id').agg(list)

/tmp/ipykernel_44860/230743473.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe[column_name] = dataframe[column_name].apply(lambda x: handle_maiyamok(x, maiyamok_exceptions))
/tmp/ipykernel_44860/230743473.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe[column_name] = dataframe[column_name].apply(lambda x: x.replace(old_word, new_word))


# Define new id instead of client_id

In [5]:
id_mapper = {id_: f'cv{str(i+1).zfill(3)}' for i, id_ in enumerate(filtered_data['client_id'].unique())}

grouped_data = {
    id_mapper[client_id]: list(zip(sentences, paths))
    for (client_id, (sentences, paths)) in grouped[['sentence', 'path']].iterrows()
}

# Moving files to the new folder

In [6]:
# Define paths
DEST_DIR = "../data/converted/commonvoice-to-virtual-vctk"
DEST_TEXT_PATH = os.path.join(DEST_DIR, "txt")
DEST_AUDIO_PATH = os.path.join(DEST_DIR, "wav32")
SRC_AUDIO_PATH = "../data/raw/cv-corpus-20.0-2024-12-06/th/clips"

# Number of worker threads (adjust based on your CPU)
NUM_WORKERS = multiprocessing.cpu_count()

def process_client_data(client_data: Tuple[str, List]) -> Dict:
    """
    Process all data for a single client
    
    Args:
        client_data: Tuple of (client_id, data)
    
    Returns:
        Dict with processing results
    """
    client_id, data = client_data
    results = {
        'client_id': client_id,
        'processed': 0,
        'failed': 0,
        'chars': set()
    }
    
    # Create client directories
    client_text_dir = os.path.join(DEST_TEXT_PATH, client_id)
    client_audio_dir = os.path.join(DEST_AUDIO_PATH, client_id)
    os.makedirs(client_text_dir, exist_ok=True)
    os.makedirs(client_audio_dir, exist_ok=True)
    
    for i, d in enumerate(data):
        # Write text file
        text_path = os.path.join(client_text_dir, f"{client_id}_{(i + 1):03d}.txt")
        with open(text_path, 'w') as f:
            f.write(d[0])
            results['chars'].update(d[0])
        
        # Convert audio file
        src_audio_path = os.path.join(SRC_AUDIO_PATH, d[1])
        dst_audio_path = os.path.join(
            client_audio_dir,
            f"{client_id}_{(i + 1):03d}_mic1.flac"
        )
        
        if convert_mp3_to_flac(src_audio_path, dst_audio_path):
            results['processed'] += 1
        else:
            results['failed'] += 1
    
    return results

# Clean and create directories
if os.path.exists(DEST_DIR):
    print("Clearing destination folder")
    shutil.rmtree(DEST_DIR)
os.makedirs(DEST_TEXT_PATH, exist_ok=True)
os.makedirs(DEST_AUDIO_PATH, exist_ok=True)

print(f"Starting parallel processing with {NUM_WORKERS} workers")

# Create progress bar for overall processing
with tqdm(total=len(grouped_data), desc="Processing clients") as pbar:
    all_chars = set()
    total_processed = 0
    total_failed = 0
    
    # Process clients in parallel
    with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        # Submit all client processing tasks
        future_to_client = {
            executor.submit(process_client_data, (client_id, data)): client_id 
            for client_id, data in grouped_data.items()
        }
        
        # Process completed tasks
        for future in concurrent.futures.as_completed(future_to_client):
            client_id = future_to_client[future]
            try:
                result = future.result()
                all_chars.update(result['chars'])
                total_processed += result['processed']
                total_failed += result['failed']
            except Exception as e:
                print(f"Client {client_id} generated an exception: {str(e)}")
                total_failed += len(grouped_data[client_id])
            pbar.update(1)

print("\nConversion Summary:")
print(f"Total files processed successfully: {total_processed}")
print(f"Total files failed: {total_failed}")
print(f"Total unique characters: {len(all_chars)}")
print("Restructuring and conversion complete")

Clearing destination folder
Starting parallel processing with 16 workers


Processing clients: 100%|██████████| 134/134 [1:04:06<00:00, 28.71s/it] 


Conversion Summary:
Total files processed successfully: 92956
Total files failed: 0
Total unique characters: 114
Restructuring and conversion complete


# Resample and trim audio

In [7]:
# Create destination directory if it doesn't exist
os.makedirs("../data/converted/commonvoice-to-virtual-vctk/wav16_silence_trimmed", exist_ok=True)

# Copy all files from wav32 to wav16_silence_trimmed
src_dir = "../data/converted/commonvoice-to-virtual-vctk/wav32"
dst_dir = "../data/converted/commonvoice-to-virtual-vctk/wav16_silence_trimmed"

recursive_copy(src_dir, dst_dir)

In [8]:
# Resample all files in wav16_silence_trimmed to 16kHz
SAMPLE_RATE = 16000
NUM_RESAMPLE_THREADS = 4

resample_audios(
  input_folders=dst_dir,
  file_ext="flac",
  sample_rate=SAMPLE_RATE,
  n_jobs=NUM_RESAMPLE_THREADS
)

Resampling the audio files...
Found 92956 files...


100%|██████████| 92956/92956 [01:28<00:00, 1053.43it/s]


Done !


In [9]:
# Trim silence at the beginning and end of each audio file
trim_silence_with_vad(
  input_folder=dst_dir,
  file_extension="flac",
)

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /home/ming/.cache/torch/hub/master.zip


Found 92956 .flac files to process


Processing files:   5%|▍         | 4586/92956 [04:27<1:11:46, 20.52it/s]

> The file ../data/converted/commonvoice-to-virtual-vctk/wav16_silence_trimmed/cv131/cv131_2196_mic1.flac probably does not have speech please check it !!


Processing files:  11%|█▏        | 10466/92956 [10:19<1:08:39, 20.02it/s]

...

> The file ../data/converted/commonvoice-to-virtual-vctk/wav16_silence_trimmed/cv132/cv132_116_mic1.flac probably does not have speech please check it !!


Processing files:  96%|█████████▌| 89091/92956 [1:30:52<02:52, 22.42it/s]

> The file ../data/converted/commonvoice-to-virtual-vctk/wav16_silence_trimmed/cv132/cv132_4774_mic1.flac probably does not have speech please check it !!


Processing files: 100%|██████████| 92956/92956 [1:34:39<00:00, 16.37it/s]


Processing complete

Found 29 files with no speech. List saved to ../data/converted/commonvoice-to-virtual-vctk/no_speech_files.txt


In [10]:
# Normalize the volume of all audio files to -27dB
normalize_audio_files(
  input_dir=dst_dir,
  extensions="flac",
  target_db=-27,
  sample_rate=16000,
)

Normalizing audio files: 100%|██████████| 92956/92956 [6:01:14<00:00,  4.29it/s]  


# New Grouping for virtual speakerId

In [11]:
# Clean and create directories
if os.path.exists(DEST_AUDIO_PATH):
    print("Clearing destination folder")
    shutil.rmtree(DEST_AUDIO_PATH)

Clearing destination folder


In [12]:
# Generate DVector using wavlm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained('microsoft/wavlm-base-plus-sv')
model = WavLMForXVector.from_pretrained('microsoft/wavlm-base-plus-sv').to(device)

if not os.path.exists('../data/converted/commonvoice-to-virtual-vctk/dvectors.pth'):
    speaker_mapping = {}
    for speaker in tqdm(os.listdir(dst_dir)):
        for audio_file in os.listdir(os.path.join(dst_dir, speaker)):
            audio_path = os.path.join(dst_dir, speaker, audio_file)
            audio_input, sr = torchaudio.load(audio_path)
            audio_input = audio_input
            audio_array = audio_input.detach().cpu().numpy()
            with torch.no_grad():
                inputs = feature_extractor(audio_array, sampling_rate=16000, return_tensors="pt")
                inputs = {k: v.to(device) for k, v in inputs.items()}
                embeddings = model(**inputs).embeddings
                embeddings = torch.nn.functional.normalize(embeddings, dim=-1).cpu()
                if embeddings.isnan().any():
                    print(f"The embedding of {audio_file} is NaN")
                    continue
            speaker_mapping[audio_file] = {}
            speaker_mapping[audio_file]['embedding'] = embeddings
            speaker_mapping[audio_file]['name'] = speaker
    torch.save(speaker_mapping, '../data/converted/commonvoice-to-virtual-vctk/dvectors.pth')
else:
    speaker_mapping = torch.load('../data/converted/commonvoice-to-virtual-vctk/dvectors.pth')

  0%|          | 0/134 [00:00<?, ?it/s]/home/ming/.cache/pypoetry/virtualenvs/speech-dataset-converter-aRtuyZwp-py3.11/lib/python3.11/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
 39%|███▉      | 52/134 [04:08<07:00,  5.13s/it]/home/ming/.cache/pypoetry/virtualenvs/speech-dataset-converter-aRtuyZwp-py3.11/lib/python3.11/site-packages/transformers/models/wavlm/modeling_wavlm.py:1833: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  std_features.append(hidden_states[i, :length].std(dim=0))


The embedding of cv044_098_mic1.flac is NaN
The embedding of cv044_124_mic1.flac is NaN


 55%|█████▌    | 74/134 [11:04<04:52,  4.88s/it]   

The embedding of cv133_8443_mic1.flac is NaN
The embedding of cv133_3365_mic1.flac is NaN


 57%|█████▋    | 77/134 [13:19<21:45, 22.91s/it]

...

 88%|████████▊ | 118/134 [16:56<01:47,  6.70s/it]

The embedding of cv054_120_mic1.flac is NaN


 96%|█████████▌| 128/134 [17:34<00:17,  2.97s/it]

The embedding of cv132_5823_mic1.flac is NaN
The embedding of cv132_737_mic1.flac is NaN


100%|██████████| 134/134 [19:39<00:00,  8.80s/it]


In [13]:
BATCH_SIZE = 10000  # Adjust based on your system's RAM

def process_speaker(name, dvector_items, verbose=False):
    total_items = len(dvector_items)
    
    # Prepare embeddings
    dvector_embeddings = np.array([value['embedding'] for _, value in dvector_items])
    dvector_embeddings = dvector_embeddings.squeeze()
            
    cosine_distances_matrix = cosine_distances(dvector_embeddings, dvector_embeddings)

    dvector_clusterer = AgglomerativeClustering(
        metric="precomputed",
        linkage="average",
        distance_threshold=1-0.85,
        n_clusters=None
    ).fit(cosine_distances_matrix)

    labels = defaultdict(int)
    for label in dvector_clusterer.labels_:
        labels[label] += 1

    if verbose: print([(key, labels[key]) for key in sorted(labels, key=labels.get, reverse=True)])

    # if value < 10 change key to -1
    for i in range(total_items):
        if labels[dvector_clusterer.labels_[i]] < 10:
            dvector_clusterer.labels_[i] = -1

    dvector_labels = dvector_clusterer.labels_
    
    concat_labels = defaultdict(list)
    cluster_stats = defaultdict(lambda: {'file_count': 0})
    
    for i in range(total_items):
        key = dvector_items[i][0]  # Extract filename (key)
        cluster_key = f'{dvector_labels[i]}'
        concat_labels[cluster_key].append(key)
        cluster_stats[cluster_key]['file_count'] += 1

    gc.collect()

    return concat_labels

# Main processing loop
speaker_dict_dvector = defaultdict(list)

for k, v in speaker_mapping.items():
    name = v['name']
    speaker_dict_dvector[name].append((k, v))

global_speaker_dict = {}
for name in tqdm(speaker_dict_dvector, desc="Processing speakers"):
    concat_labels = process_speaker(name, speaker_dict_dvector[name])
    for cluster_key, cluster_files in concat_labels.items():
        if cluster_key == '-1':
            continue
        global_speaker_dict[f'v{name}_{cluster_key}'] = cluster_files

json.dump(global_speaker_dict, open('../data/converted/commonvoice-to-virtual-vctk/virtual_speaker_mapping.json', 'w'), indent=4)

Processing speakers: 100%|██████████| 134/134 [01:27<00:00,  1.54it/s]


In [14]:
for speaker in tqdm(global_speaker_dict, desc="Processing speakers"):
    speaker_name = speaker
    speaker_audio_files = global_speaker_dict[speaker]
    speaker_text_files = [os.path.join(DEST_TEXT_PATH, f"{audio_file.split('_')[0]}/{'_'.join(audio_file.split('.')[0].split('_')[:2])}.txt") for audio_file in speaker_audio_files]

    speaker_audio_dir = os.path.join("../data/converted/commonvoice-to-virtual-vctk/wav16_silence_trimmed", speaker_name)
    os.makedirs(speaker_audio_dir, exist_ok=True)

    for audio_file in speaker_audio_files:
        src_audio_path = os.path.join(dst_dir, audio_file.split('_')[0], audio_file)
        dst_audio_path = os.path.join(speaker_audio_dir, audio_file)
        shutil.copy(src_audio_path, dst_audio_path)

    speaker_text_dir = os.path.join("../data/converted/commonvoice-to-virtual-vctk/txt", speaker_name)
    os.makedirs(speaker_text_dir, exist_ok=True)
    
    for text_file in speaker_text_files:
        src_text_path = text_file
        dst_text_path = os.path.join(speaker_text_dir, os.path.basename(text_file))
        shutil.copy(src_text_path, dst_text_path)

Processing speakers: 100%|██████████| 361/361 [00:20<00:00, 17.34it/s]


In [15]:
# remove folders with cv prefix in dst_dir
for speaker in os.listdir(dst_dir):
    if speaker.startswith('cv'):
        shutil.rmtree(os.path.join(dst_dir, speaker))

for speaker in os.listdir("../data/converted/commonvoice-to-virtual-vctk/txt"):
    if speaker.startswith('cv'):
        shutil.rmtree(os.path.join("../data/converted/commonvoice-to-virtual-vctk/txt", speaker))

# Manual Clean Data

In [2]:
# hand labelled by @Thanakorn and @Natyanin
verified_speakers = ['vcv080_0', 'vcv080_1', 'vcv038_0', 'vcv038_4', 'vcv109_2', 'vcv058_0', 'vcv036_0', 'vcv016_2', 'vcv005_0', 'vcv034_0', 'vcv078_0', 'vcv078_1', 'vcv087_1', 'vcv087_0', 'vcv006_1', 'vcv131_1', 'vcv131_4', 'vcv131_7', 'vcv061_2', 'vcv127_9', 'vcv127_1', 'vcv127_48', 'vcv127_82', 'vcv127_23', 'vcv127_0', 'vcv127_3', 'vcv127_12', 'vcv020_0', 'vcv097_1', 'vcv050_0', 'vcv046_0', 'vcv110_0', 'vcv033_0', 'vcv003_0', 'vcv096_5', 'vcv096_3', 'vcv084_0', 'vcv072_0', 'vcv072_2', 'vcv047_0', 'vcv047_2', 'vcv011_2', 'vcv010_1', 'vcv024_0', 'vcv053_0', 'vcv068_0', 'vcv055_3', 'vcv055_6', 'vcv055_0', 'vcv055_1', 'vcv013_0', 'vcv095_0', 'vcv095_3', 'vcv095_2', 'vcv075_1', 'vcv117_1', 'vcv117_0', 'vcv074_0', 'vcv014_1', 'vcv125_15', 'vcv125_19', 'vcv125_16', 'vcv125_17', 'vcv125_4', 'vcv125_0', 'vcv125_5', 'vcv125_9', 'vcv125_20', 'vcv023_2', 'vcv041_0', 'vcv041_2', 'vcv123_2', 'vcv123_1', 'vcv123_3', 'vcv123_6', 'vcv017_1', 'vcv044_2', 'vcv030_2', 'vcv067_0', 'vcv085_2', 'vcv085_1', 'vcv048_0', 'vcv083_2', 'vcv059_0', 'vcv134_4', 'vcv134_25', 'vcv134_6', 'vcv134_15', 'vcv134_51', 'vcv134_8', 'vcv134_28', 'vcv134_14', 'vcv134_5', 'vcv134_23', 'vcv134_2', 'vcv134_16', 'vcv134_44', 'vcv134_60', 'vcv134_12', 'vcv134_48', 'vcv134_13', 'vcv134_63', 'vcv134_56', 'vcv134_20', 'vcv134_52', 'vcv134_10', 'vcv134_7', 'vcv134_9', 'vcv134_3', 'vcv134_40', 'vcv134_26', 'vcv134_45', 'vcv134_0', 'vcv134_47', 'vcv134_1', 'vcv134_112', 'vcv134_43', 'vcv007_1', 'vcv129_1', 'vcv129_24', 'vcv129_3', 'vcv129_13', 'vcv129_6', 'vcv129_8', 'vcv129_2', 'vcv129_19', 'vcv027_0', 'vcv062_0', 'vcv019_0', 'vcv025_2', 'vcv025_0', 'vcv035_0', 'vcv004_0', 'vcv004_1', 'vcv103_0', 'vcv028_0', 'vcv029_6', 'vcv029_1', 'vcv118_1', 'vcv070_0', 'vcv133_21', 'vcv133_3', 'vcv133_10', 'vcv133_16', 'vcv133_19', 'vcv133_4', 'vcv133_22', 'vcv133_2', 'vcv133_32', 'vcv133_26', 'vcv133_1', 'vcv133_9', 'vcv133_8', 'vcv133_31', 'vcv133_14', 'vcv133_0', 'vcv133_6', 'vcv008_1', 'vcv115_1', 'vcv115_0', 'vcv115_2', 'vcv115_13', 'vcv099_0', 'vcv126_2', 'vcv126_0', 'vcv126_6', 'vcv126_11', 'vcv126_3', 'vcv126_8', 'vcv126_19', 'vcv126_20', 'vcv126_14', 'vcv002_0', 'vcv124_1', 'vcv124_8', 'vcv124_16', 'vcv092_1', 'vcv100_1', 'vcv100_4', 'vcv001_0', 'vcv064_0', 'vcv089_3', 'vcv091_0', 'vcv120_2', 'vcv120_4', 'vcv021_1', 'vcv052_0', 'vcv040_1', 'vcv086_0', 'vcv042_0', 'vcv032_14', 'vcv076_0', 'vcv113_0', 'vcv113_2', 'vcv105_0', 'vcv104_0', 'vcv104_1', 'vcv051_0', 'vcv051_1', 'vcv114_6', 'vcv114_4', 'vcv114_2', 'vcv079_0', 'vcv079_3', 'vcv102_4', 'vcv102_0', 'vcv071_3', 'vcv071_7', 'vcv009_0', 'vcv015_0', 'vcv063_0', 'vcv107_2', 'vcv082_0', 'vcv082_6', 'vcv022_0', 'vcv098_6', 'vcv088_0', 'vcv116_1', 'vcv049_0', 'vcv130_1', 'vcv130_2', 'vcv130_5', 'vcv130_12', 'vcv130_27', 'vcv130_9', 'vcv130_15', 'vcv130_19', 'vcv073_0', 'vcv026_0', 'vcv054_4', 'vcv108_0', 'vcv108_2', 'vcv108_4', 'vcv108_5', 'vcv108_1', 'vcv093_0', 'vcv057_5', 'vcv057_0', 'vcv111_0', 'vcv111_6', 'vcv112_2', 'vcv112_1', 'vcv112_3', 'vcv081_0', 'vcv012_0', 'vcv043_0', 'vcv060_0', 'vcv132_17', 'vcv132_20', 'vcv132_16', 'vcv132_34', 'vcv132_54', 'vcv132_0', 'vcv132_212', 'vcv132_26', 'vcv132_1', 'vcv132_31', 'vcv132_150', 'vcv132_36', 'vcv132_6', 'vcv132_37', 'vcv132_64', 'vcv132_42', 'vcv132_156', 'vcv132_67', 'vcv132_50', 'vcv132_252', 'vcv132_38', 'vcv132_48', 'vcv132_148', 'vcv132_8', 'vcv132_10', 'vcv132_92', 'vcv132_70', 'vcv132_125', 'vcv132_29', 'vcv132_25', 'vcv132_57', 'vcv132_82', 'vcv132_7', 'vcv132_138', 'vcv132_104', 'vcv132_9', 'vcv132_187', 'vcv132_69', 'vcv132_103', 'vcv132_68', 'vcv122_1', 'vcv122_9', 'vcv122_10', 'vcv090_0', 'vcv018_0', 'vcv128_1', 'vcv128_3', 'vcv128_0', 'vcv065_0']

for speaker in os.listdir("../data/converted/commonvoice-to-virtual-vctk/txt"):
    if speaker not in verified_speakers:
        shutil.rmtree(os.path.join("../data/converted/commonvoice-to-virtual-vctk/txt", speaker))
        shutil.rmtree(os.path.join("../data/converted/commonvoice-to-virtual-vctk/wav16_silence_trimmed", speaker))

# Output Stat

In [4]:
# @T. Wong
# Path pattern
path_pattern = "../data/converted/commonvoice-to-virtual-vctk/wav16_silence_trimmed/**/*.flac"

# Get all flac files matching the pattern
flac_files = glob.glob(path_pattern, recursive=True)

if not flac_files:
    print(f"No FLAC files found matching pattern: {path_pattern}")
else:
    print(f"Found {len(flac_files)} FLAC files. Processing...")
    
    # Calculate total duration
    total_duration_seconds = 0
    
    # Track speaker folders
    speaker_folders = set()
    
    # Extract base directory for later use in calculating speaker directories
    base_dir = os.path.normpath("../data/converted/commonvoice-to-virtual-vctk/wav16_silence_trimmed")
    
    # Use tqdm for progress bar
    for flac_file in tqdm(flac_files):
        try:
            # Get audio info
            info = sf.info(flac_file)
            total_duration_seconds += info.duration
            
            # Extract speaker folder - take the directory right after wav16_silence_trimmed/
            rel_path = os.path.relpath(os.path.dirname(flac_file), base_dir)
            if '/' in rel_path:
                speaker = rel_path.split('/')[0]  # First directory is the speaker
            else:
                speaker = rel_path  # If there's no further nesting
                
            speaker_folders.add(speaker)
            
        except Exception as e:
            print(f"Error processing {flac_file}: {e}")
    
    # Convert to hours
    total_duration_hours = total_duration_seconds / 3600
    
    # Print results
    print(f"\nTotal duration: {total_duration_hours:.2f} hours")
    print(f"                ({total_duration_hours*60:.2f} minutes)")
    print(f"                ({total_duration_hours*3600:.2f} seconds)")
    
    # Print only the number of speakers
    print(f"Number of speakers: {len(speaker_folders)}")

Found 83111 FLAC files. Processing...


100%|██████████| 83111/83111 [00:06<00:00, 13783.72it/s]


Total duration: 61.50 hours
                (3690.00 minutes)
                (221400.15 seconds)
Number of speakers: 296
